# Learn LangChain Basics

This notebook introduces the core building blocks of LangChain, including LLMs, prompt templates, chains, memory, document retrieval, and simple question-answering flows.

> Note: This notebook is written for LangChain v0.0.x or later. Adjust import paths if you use a different version.

## Install dependencies

Install the required packages if they are not already installed. In a Jupyter environment, you can run:

```bash
pip install langchain openai tiktoken faiss-cpu python-dotenv
```

## Setup your OpenAI API key

LangChain uses an LLM provider. For this notebook we use OpenAI, so set the environment variable `OPENAI_API_KEY` before running the code.

```python
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from a .env file if present
openai_key = os.getenv('OPENAI_API_KEY')
assert openai_key is not None, 'Set OPENAI_API_KEY in your environment.'
```

## Create an LLM instance

The LLM is the core component that generates text. We instantiate the OpenAI model and call it with a prompt.

In [ ]:
from langchain import OpenAI

llm = OpenAI(model_name='gpt-3.5-turbo', temperature=0.7)
response = llm('Write a short poem about learning LangChain.')
print(response)

## Use a prompt template

Prompt templates make prompts reusable and easier to maintain. We can define variables and then format the template with values.

In [ ]:
from langchain import PromptTemplate
from langchain import LLMChain

template = PromptTemplate(
    input_variables=['topic'],
    template='Write a friendly paragraph explaining why {topic} is useful.',
)

chain = LLMChain(llm=llm, prompt=template)
result = chain.run({'topic': 'LangChain'})
print(result)

## Build a simple chain

Chains connect prompts and model calls into a composed workflow. This is useful when you want multiple steps or input/output transformations.

In [ ]:
from langchain import PromptTemplate, LLMChain, SimpleSequentialChain

first_prompt = PromptTemplate(
    input_variables=['product'],
    template='Describe the main benefit of {product}.',
)
second_prompt = PromptTemplate(
    input_variables=['description'],
    template='Rewrite the following description as a tweet: {description}',
)

first_chain = LLMChain(llm=llm, prompt=first_prompt, output_key='description')
second_chain = LLMChain(llm=llm, prompt=second_prompt)

overall_chain = SimpleSequentialChain(chains=[first_chain, second_chain], verbose=True)
tweet = overall_chain.run('LangChain tutorials')
print(tweet)

## Add conversation memory

Memory stores context across interactions so the model can follow the conversation state. Here we use a simple buffer memory.

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key='chat_history', input_key='question')

chat_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate(
        input_variables=['chat_history', 'question'],
        template='You are a helpful assistant.
Chat history:
{chat_history}
Human: {question}
Assistant:',
    ),
    memory=memory,
    verbose=True,
)

print(chat_chain.run({'question': 'What is LangChain?'}))
print(chat_chain.run({'question': 'Why is it useful for building AI applications?'}))

## Load documents and build a retriever

LangChain can retrieve information from documents using embeddings and a vector store. The example below uses a simple in-memory FAISS index.

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# Example texts
texts = [
    'LangChain helps you build applications with large language models.',
    'LangChain provides chains, prompts, agents, and memory for orchestrating model calls.',
    'Vector stores can search for relevant documents using embeddings.',
]

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=0)
docs = text_splitter.split_documents([{'page_content': text} for text in texts])

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

query = 'How does LangChain use vector stores?'
results = vectorstore.similarity_search(query, k=2)
for i, doc in enumerate(results, 1):
    print(f'
Result {i}:')
    print(doc.page_content)

## Build a retrieval QA chain

Combine a retriever with an LLM to answer questions using the document content.

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=vectorstore.as_retriever(),
)

answer = qa_chain.run('What can LangChain do with documents?')
print(answer)

## Next steps

- Explore LangChain agents and tool use.
- Experiment with different embeddings and vector stores.
- Add real document loaders for PDFs, websites, or databases.